# Bước 1: Cài đặt thư viện
Chạy ô này để cài đặt thư viện giao tiếp với Azure và kiểm tra xem máy có nhận GPU không.

In [ ]:
!pip install azure-storage-blob
!nvidia-smi

# Bước 2: Khai báo cấu hình
Điền thông tin kết nối Azure của bạn vào đây.

In [ ]:
AZURE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=aicvideos;AccountKey=ZLdH5hcKb213WiCzOLCmfLrPRWR3rVkDdlfwV5iXr2snbY9lAMAJc9ZlHm9salxNTAvhcaNVJh0N+ASt+nNulQ==;EndpointSuffix=core.windows.net"
CONTAINER_NAME = "videos"

# Đường dẫn tĩnh Batch 1 mà bạn đang dùng
TARGET_DIR = "/kaggle/input/datasets/aresusayhi/ai-challenge-2025/Videos/Videos"

# Nơi lưu file tạm trước khi up lên Azure (sẽ bị xóa ngay sau khi up để trống ổ)
TEMP_DIR = "/kaggle/working/proxy_videos"

# Bước 3: Hàm xử lý FFmpeg (720p + Fast Start)
Hàm này copy giữ nguyên 100% chất lượng gốc, chỉ chèn cờ Fast Start.

In [ ]:
import os
import subprocess
from pathlib import Path

Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)

def process_video_ffmpeg(input_path, output_path):
    """
    Giữ nguyên 100% chất lượng gốc (không giảm dung lượng, không giảm độ phân giải).
    Chỉ chèn thêm cờ faststart để sửa lỗi tua video bị đơ.
    """
    command = [
        "ffmpeg", "-y",             # -y: Ghi đè file nếu có
        "-i", str(input_path),
        "-c", "copy",               # Copy y nguyên luồng hình ảnh/âm thanh gốc, KHÔNG nén lại
        "-movflags", "faststart",   # TỐI QUAN TRỌNG: Dời mục lục lên đầu để tua nhanh
        str(output_path)
    ]
    
    try:
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        return True
    except subprocess.CalledProcessError as e:
        print(f"Lỗi FFmpeg: {e.stderr.decode('utf-8', errors='ignore')}")
        return False


# Bước 4: Khởi chạy Upload hàng loạt (Theo cấu trúc Batch 1)
Truy tìm 16 thư mục con y hệt như file azure_data.ipynb cũ của bạn.

In [ ]:
import os
import subprocess
from pathlib import Path

Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)

def process_video_ffmpeg(input_path, output_path):
    """
    Giữ nguyên 100% chất lượng gốc (không giảm dung lượng, không giảm độ phân giải).
    Chỉ chèn thêm cờ faststart để sửa lỗi tua video bị đơ.
    """
    command = [
        "ffmpeg", "-y",             # -y: Ghi đè file nếu có
        "-i", str(input_path),
        "-c", "copy",               # Copy y nguyên luồng hình ảnh/âm thanh gốc, KHÔNG nén lại
        "-movflags", "faststart",   # TỐI QUAN TRỌNG: Dời mục lục lên đầu để tua nhanh
        str(output_path)
    ]
    
    try:
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        return True
    except subprocess.CalledProcessError as e:
        print(f"Lỗi FFmpeg: {e.stderr.decode('utf-8', errors='ignore')}")
        return False
